In [1]:

from google.colab import files

uploaded = files.upload()


Saving archive (2).zip to archive (2).zip


In [2]:

import zipfile
import os

zip_path = "archive (2).zip"
extract_path = "/content/uber_dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

for root, dirs, files_list in os.walk(extract_path):
    for file in files_list:
        print(os.path.join(root, file))


/content/uber_dataset/uber_trips_dataset_50k.csv


In [3]:

import glob
from pyspark.sql import SparkSession

# Start or reuse the Spark session
spark = (
    SparkSession.builder
    .appName("RideSharingETL")
    .getOrCreate()
)

# Locate the extracted CSV file
csv_files = glob.glob(
    "/content/uber_dataset/**/*.csv",
    recursive=True
)

print("CSV files found:", csv_files)

# Load the dataset into a Spark DataFrame
csv_path = csv_files[0]

rides_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

print("Total records:", rides_df.count())
print("Total columns:", len(rides_df.columns))


CSV files found: ['/content/uber_dataset/uber_trips_dataset_50k.csv']
Total records: 50000
Total columns: 14


In [4]:

# Display column names and data types
rides_df.printSchema()

# Show the first 5 records
rides_df.show(5, truncate=False)

# Display column names separately
print("Column names:")
print(rides_df.columns)


root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- rider_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- pickup_lat: double (nullable = true)
 |-- pickup_lng: double (nullable = true)
 |-- drop_lat: double (nullable = true)
 |-- drop_lng: double (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- pickup_time: timestamp (nullable = true)
 |-- drop_time: timestamp (nullable = true)

+-------+---------+--------+-------------+-----------------+-------------------+-----------------+-------------------+-----------+-----------+---------+--------------+-------------------+---------------------+
|trip_id|driver_id|rider_id|city         |pickup_lat       |pickup_lng         |drop_lat         |drop_lng           |distance_km|fare_amount|status   |payment_method|pickup_time        |drop_time     

In [6]:

from pyspark.sql import functions as F

# Calculate trip duration in minutes
rides_df = rides_df.withColumn(
    "trip_duration_minutes",
    (
        F.col("drop_time").cast("long")
        - F.col("pickup_time").cast("long")
    ) / 60
)

# Separate invalid rides for error handling
invalid_rides = rides_df.filter(
    F.col("trip_duration_minutes").isNull()
    | (F.col("trip_duration_minutes") <= 0)
)

valid_rides = rides_df.filter(
    F.col("trip_duration_minutes") > 0
)

print("Total rides:", rides_df.count())
print("Valid rides:", valid_rides.count())
print("Invalid rides:", invalid_rides.count())

# Display calculated durations
valid_rides.select(
    "trip_id",
    "driver_id",
    "pickup_time",
    "drop_time",
    "trip_duration_minutes"
).show(5, truncate=False)


Total rides: 50000
Valid rides: 49997
Invalid rides: 3
+-------+---------+-------------------+---------------------+---------------------+
|trip_id|driver_id|pickup_time        |drop_time            |trip_duration_minutes|
+-------+---------+-------------------+---------------------+---------------------+
|1      |8270     |2023-01-01 00:00:00|2023-01-01 00:08:54.6|8.9                  |
|2      |1860     |2023-01-01 00:01:00|2023-01-01 00:26:17.4|25.283333333333335   |
|3      |6390     |2023-01-01 00:02:00|2023-01-01 00:18:22.8|16.366666666666667   |
|4      |6191     |2023-01-01 00:03:00|2023-01-01 00:22:49.8|19.816666666666666   |
|5      |6734     |2023-01-01 00:04:00|2023-01-01 00:35:30  |31.5                 |
+-------+---------+-------------------+---------------------+---------------------+
only showing top 5 rows


In [7]:

from pyspark.sql import functions as F

# Calculate trip duration in minutes
rides_df = rides_df.withColumn(
    "trip_duration_minutes",
    (
        F.col("drop_time").cast("long")
        - F.col("pickup_time").cast("long")
    ) / 60
)

# Add an error reason to invalid rides
invalid_rides = (
    rides_df
    .withColumn(
        "error_reason",
        F.when(
            F.col("trip_duration_minutes").isNull(),
            "Missing or invalid pickup/drop timestamp"
        ).when(
            F.col("trip_duration_minutes") <= 0,
            "Zero or negative trip duration"
        )
    )
    .filter(F.col("error_reason").isNotNull())
)

# Keep rides with valid duration
valid_rides = rides_df.filter(
    F.col("trip_duration_minutes").isNotNull()
    & (F.col("trip_duration_minutes") > 0)
)

print("Total rides:", rides_df.count())
print("Valid rides:", valid_rides.count())
print("Rejected rides:", invalid_rides.count())

# Preview rejected records
invalid_rides.select(
    "trip_id",
    "driver_id",
    "trip_duration_minutes",
    "error_reason"
).show(10, truncate=False)


Total rides: 50000
Valid rides: 49997
Rejected rides: 3
+-------+---------+---------------------+------------------------------+
|trip_id|driver_id|trip_duration_minutes|error_reason                  |
+-------+---------+---------------------+------------------------------+
|39362  |2853     |0.0                  |Zero or negative trip duration|
|39375  |1857     |0.0                  |Zero or negative trip duration|
|44303  |3552     |0.0                  |Zero or negative trip duration|
+-------+---------+---------------------+------------------------------+



In [8]:

# Create an output folder
import os

os.makedirs("/content/ride_sharing_output", exist_ok=True)

# Save rejected rides as a CSV error log
invalid_rides.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/ride_sharing_output/error_log")

print("Error log saved successfully!")

# Check the rejected records
print("Rejected ride count:", invalid_rides.count())


Error log saved successfully!
Rejected ride count: 3


In [9]:

# Save valid ride records as JSON telemetry
valid_rides.write \
    .mode("overwrite") \
    .json("/content/ride_sharing_output/ride_telemetry_json")

print("Ride telemetry JSON saved successfully!")

# Preview the saved JSON data
display(
    spark.read.json(
        "/content/ride_sharing_output/ride_telemetry_json"
    ).limit(5).toPandas()
)


Ride telemetry JSON saved successfully!


,city,distance_km,driver_id,drop_lat,drop_lng,drop_time,fare_amount,payment_method,pickup_lat,pickup_lng,pickup_time,rider_id,status,trip_duration_minutes,trip_id
0,San Francisco,2.97,8270,37.173652,-77.619934,2023-01-01T00:08:54.600Z,10.71,Wallet,37.170931,-77.586479,2023-01-01T00:00:00.000Z,10683,Completed,8.900000,1
1,Boston,8.43,1860,38.937464,-108.558727,2023-01-01T00:26:17.400Z,22.41,UPI,38.898127,-108.582977,2023-01-01T00:01:00.000Z,44743,Completed,25.283333,2
2,San Francisco,5.46,6390,38.821702,-89.896435,2023-01-01T00:18:22.800Z,12.91,Cash,38.814571,-89.942603,2023-01-01T00:02:00.000Z,75839,Completed,16.366667,3
3,New York,6.61,6191,37.301375,-75.317488,2023-01-01T00:22:49.800Z,15.70,Wallet,37.295906,-75.328844,2023-01-01T00:03:00.000Z,22189,Completed,19.816667,4
4,Seattle,10.50,6734,38.992088,-121.467904,2023-01-01T00:35:30.000Z,19.15,Wallet,38.972395,-121.482913,2023-01-01T00:04:00.000Z,61104,Completed,31.500000,5


In [10]:

import subprocess

for command in [
    ["java", "-version"],
    ["bash", "-lc", "which hdfs || echo 'HDFS command not found'"],
    ["bash", "-lc", "which hadoop || echo 'Hadoop command not found'"]
]:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )
    print("$", " ".join(command))
    print(result.stdout or result.stderr)
    print("-" * 40)


$ java -version
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

----------------------------------------
$ bash -lc which hdfs || echo 'HDFS command not found'
HDFS command not found

----------------------------------------
$ bash -lc which hadoop || echo 'Hadoop command not found'
Hadoop command not found

----------------------------------------


In [11]:

import os
import subprocess

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

commands = [
    "ps -ef | grep -i '[N]amenode'",
    "ps -ef | grep -i '[D]atanode'",
    "find / -type f -name hdfs 2>/dev/null | head -10",
    "find / -type f -name start-dfs.sh 2>/dev/null | head -10"
]

for cmd in commands:
    print(f"\n$ {cmd}")
    result = subprocess.run(
        ["bash", "-lc", cmd],
        capture_output=True,
        text=True,
        timeout=60
    )
    print(result.stdout or "No matching process or file found.")


HADOOP_HOME: None
JAVA_HOME: None

$ ps -ef | grep -i '[N]amenode'
No matching process or file found.

$ ps -ef | grep -i '[D]atanode'
No matching process or file found.

$ find / -type f -name hdfs 2>/dev/null | head -10
No matching process or file found.

$ find / -type f -name start-dfs.sh 2>/dev/null | head -10
No matching process or file found.


In [12]:

# Install Java and download Hadoop
!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk-headless

!wget -q https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz -O /content/hadoop.tar.gz

!tar -xzf /content/hadoop.tar.gz -C /content/

print("Java and Hadoop installation files are ready.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.32.1+1-1ubuntu1~24.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.32.1+1-1ubuntu1~24.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.32.1+1-1ubuntu1~24.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /us

In [13]:

import os
import subprocess

os.environ["JAVA_HOME"] = subprocess.check_output(
    ["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"],
    text=True
).strip()

os.environ["HADOOP_HOME"] = "/content/hadoop-3.4.2"
os.environ["PATH"] += ":" + os.environ["HADOOP_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/sbin"

!hadoop version
!hdfs version


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /content/hadoop-3.4.2/share/hadoop/common/hadoop-common-3.4.2.jar
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detec

In [14]:

import os
from pathlib import Path

HADOOP_HOME = "/content/hadoop-3.4.2"

java_path = os.path.realpath(
    os.popen("which java").read().strip()
)
JAVA_HOME = str(Path(java_path).parent.parent)

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:"
    + os.environ["PATH"]
)

# Configure Hadoop to use the current Java installation
hadoop_env = f"{HADOOP_HOME}/etc/hadoop/hadoop-env.sh"

with open(hadoop_env, "a") as f:
    f.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")

# Configure single-node HDFS
core_site = f"""<?xml version="1.0"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
"""

hdfs_site = f"""<?xml version="1.0"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///content/hadoop_data/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:///content/hadoop_data/datanode</value>
  </property>
</configuration>
"""

config_dir = f"{HADOOP_HOME}/etc/hadoop"

with open(f"{config_dir}/core-site.xml", "w") as f:
    f.write(core_site)

with open(f"{config_dir}/hdfs-site.xml", "w") as f:
    f.write(hdfs_site)

os.makedirs("/content/hadoop_data/namenode", exist_ok=True)
os.makedirs("/content/hadoop_data/datanode", exist_ok=True)

print("JAVA_HOME:", JAVA_HOME)
print("HDFS configuration files created.")


JAVA_HOME: /usr/lib/jvm/java-21-openjdk-amd64
HDFS configuration files created.


In [15]:

import subprocess
import os

env = os.environ.copy()
env["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
env["HADOOP_HOME"] = "/content/hadoop-3.4.2"
env["PATH"] = (
    "/content/hadoop-3.4.2/bin:"
    "/content/hadoop-3.4.2/sbin:"
    + env["PATH"]
)

result = subprocess.run(
    ["hdfs", "namenode", "-format", "-force"],
    env=env,
    capture_output=True,
    text=True,
    timeout=120
)

print(result.stdout[-3000:])
print(result.stderr[-3000:])

print("Formatting command exit code:", result.returncode)


[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

abled? true
2026-10-09 08:42:06,437 INFO namenode.NameNode: Caching file names occurring more than 10 times
2026-10-09 08:42:06,442 INFO snapshot.SnapshotManager: Loaded config captureOpenFiles: false, skipCaptureAccessTimeOnlyChange: false, snapshotDiffAllowSnapRootDescendant: true, maxSnapshotFSLimit: 65536, maxSnapshotLimit: 65536
2026-10-09 08:42:06,442 INFO snapshot.SnapshotManager: dfs.namenode.snapshot.deletion.ordered = false
2026-10-09 08:42:06,444 INFO snapshot.SnapshotManager: SkipList is disabled
2026-10-09 08:42:06,448 INFO util.GSet: Computing capacity for map cachedBlocks
2026-10-09 08:42:06,448 INFO util.GSet: VM type       = 64-bit
2026-10-09 08

In [16]:

import os
import subprocess
import time

env = os.environ.copy()
env["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
env["HADOOP_HOME"] = "/content/hadoop-3.4.2"
env["PATH"] = (
    "/content/hadoop-3.4.2/bin:"
    "/content/hadoop-3.4.2/sbin:"
    + env["PATH"]
)

# Start the NameNode
nn = subprocess.run(
    ["hdfs", "--daemon", "start", "namenode"],
    env=env, capture_output=True, text=True
)
print("NameNode:", nn.stdout, nn.stderr)

time.sleep(3)

# Start the DataNode
dn = subprocess.run(
    ["hdfs", "--daemon", "start", "datanode"],
    env=env, capture_output=True, text=True
)
print("DataNode:", dn.stdout, dn.stderr)

time.sleep(5)

# Check HDFS status
report = subprocess.run(
    ["hdfs", "dfsadmin", "-report"],
    env=env, capture_output=True, text=True,
    timeout=30
)
print(report.stdout)
print(report.stderr)
print("Report exit code:", report.returncode)


NameNode:  
DataNode:  
[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Configured Capacity: 115658190848 (107.72 GB)
Present Capacity: 90291195904 (84.09 GB)
DFS Remaining: 90291171328 (84.09 GB)
DFS Used: 24576 (24 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

----------------------------

In [17]:

import os
import subprocess

env = os.environ.copy()
env["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
env["HADOOP_HOME"] = "/content/hadoop-3.4.2"
env["PATH"] = (
    "/content/hadoop-3.4.2/bin:"
    "/content/hadoop-3.4.2/sbin:"
    + env["PATH"]
)

# Create directories in HDFS
subprocess.run(
    ["hdfs", "dfs", "-mkdir", "-p", "/ride_sharing/ride_telemetry"],
    env=env, check=True
)

# Upload the JSON output from local storage to HDFS
subprocess.run(
    [
        "hdfs", "dfs", "-put", "-f",
        "/content/ride_sharing_output/ride_telemetry_json",
        "/ride_sharing/ride_telemetry/"
    ],
    env=env, check=True
)

# Verify the uploaded files
subprocess.run(
    ["hdfs", "dfs", "-ls", "-R", "/ride_sharing"],
    env=env, check=True
)


CompletedProcess(args=['hdfs', 'dfs', '-ls', '-R', '/ride_sharing'], returncode=0)

In [18]:
!hdfs dfs -ls -R /ride_sharing/ride_telemetry

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
drwxr-xr-x   - root supergroup          0 2026-10-09 08:43 /ride_sharing/ride_telemetry/ride_telemetry_json
-rw-r--r--   1 root supergroup          0 2026-10-09 08:43 /ride_sharing/ride_telemetry/ride_telemetry_json/_SUCCESS
-rw-r--r--   1 root supergroup   14545286 2026-10-09 08:43 /ride_sharing/ride_telemetry/ride_telemetry_json/part-00000-4958fe95-413f-49a1-9ee3-79e7b3f574df-c000.json
-rw-r--r--   1 root supergroup    5197270 2026-10-09 08:43 /ride_sharing/ride_telemetry/ride_telemetry_json/part-00001-4958fe95-413f-49a1-9ee3-79e7b3f574df-c000.json
